In [ ]:
import os, re, json, gc, itertools
import time
from datetime import datetime

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

from sklearn import metrics
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold

import joblib
import xgboost as xgb
import lightgbm as lgb


from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, accuracy_score


# preprocessing

In [ ]:
# load data
train = pd.read_csv('data/second_round_training_data.csv')
test = pd.read_csv('data/second_round_testing_data.csv')

In [ ]:
# train ?? target
target = train.pop('Quality_label')

In [ ]:
# target ??
target = (target.astype("category")
          .cat.set_categories(["Fail", "Pass", "Good", "Excellent"])
          .cat.codes)
target = target.rename('target')

In [ ]:
# train ?? parameter ? attribute
col_attr=[f'Attribute{x}' for x in range(1,11)]
attr = train[col_attr]
train = train.drop(columns=col_attr)

In [ ]:
# test ?? group
group = test.pop('Group')
group = group.rename('group')

In [ ]:
# test ??????,? train ????
col_test = [f"Parameter{x}" for x in range(1, 11)]
test = test[col_test]

In [ ]:
# ????
colnames = [f'p{i}' for i in range(1,11)]
train.columns = colnames
test.columns = colnames
colnames = [f'a{i}' for i in range(1,11)]
attr.columns = colnames

# FE

In [ ]:
# rounding
round_fmt = {1:11,2:11,3:11,4:11,
             5:8,6:8,7:8,8:8,9:8,10:8}
for i,r in round_fmt.items():
    train[f'p{i}']=train[f'p{i}'].round(r)
    test[f'p{i}']=test[f'p{i}'].round(r)

In [ ]:
# transform log2
# ?????????????,?????????
# ????????????
train = train.apply(np.log2)
attr = attr.apply(np.log2)
test = test.apply(np.log2)

In [ ]:
# ?? P5-P10
train = train.drop(columns=[f'p{i}' for i in range(1,5)])
test = test.drop(columns=[f'p{i}' for i in range(1,5)])

In [ ]:
print(train.shape,test.shape)
test['p9_isnull']=test['p9'].isna()
test_na=test[test['p9_isnull']==False]
lbl=LabelEncoder()
lbl.fit(list(test_na['p9'])+list(train['p9']))
test_na['p9']=lbl.transform(test_na['p9'])
train['p9']=lbl.transform(train['p9'])
test.loc[test['p9_isnull']==False,'p9']=test_na
del test_na

# predict A4-A6 and P9

## P9

In [ ]:
test['p9_isnull']=test['p9'].isna()
test_not_na=test[test['p9_isnull']==False]
lbl=LabelEncoder()
lbl.fit(list(test_not_na['p9'])+list(train['p9']))
test_not_na['p9']=lbl.transform(test_not_na['p9'])
train['p9']=lbl.transform(train['p9'])
test.loc[test['p9_isnull']==False,'p9']=test_not_na
del test_not_na

In [ ]:
parameter='p9'

In [ ]:
p9=train['p9']
train=train.drop(columns='p9')
df=train.join(p9)

In [ ]:
test_not_na=test[test['p9_isnull']==True]

In [ ]:
test_not_na=test_not_na.drop(columns=['p9','p9_isnull'])

### LGB

In [ ]:
best_hp={'boosting_type': 'gbdt',
 'num_leaves': 111,
 'max_depth': -1,
 'learning_rate': 0.1,
 'n_estimators': 639,
 'subsample_for_bin': 200000,
 'objective': 'multiclass',
 'min_split_gain': 0,
 'min_child_weight': 0.001,
 'min_child_samples': 5,
 'subsample': 1.0,
 'subsample_freq': 0,
 'colsample_bytree': 1,
 'reg_alpha': 0.20174465129035402,
 'reg_lambda': 0.18200878389736302,
 'n_jobs': -1,
 'silent': True,
 'importance_type': 'split',
 'num_class': 17}

In [ ]:
m = lgb.LGBMClassifier().set_params(**best_hp)
m.fit(train,p9)

In [ ]:
res=m.predict(test_not_na)

In [ ]:
test.loc[test['p9_isnull']==True,'p9']=res

In [ ]:
test['p9']=test['p9'].astype(int)
test=test.drop(columns='p9_isnull')

In [ ]:
p10=train['p10']
train=train.drop(columns='p10')
train=train.join(p9)
train=train.join(p10)

In [ ]:
train['p9'].min()

## A4

In [ ]:
attribute='a4'
df = train.join(attr[attribute])

In [ ]:
best_hp={'max_depth': 5,
 'learning_rate': 0.1,
 'n_estimators': 571,
 'verbosity': 1,
 'objective': 'reg:squarederror',
 'booster': 'dart',
 'n_jobs': -1,
 'gamma': 0,
 'min_child_weight': 1,
 'max_delta_step': 0,
 'subsample': 1,
 'colsample_bytree': 1,
 'colsample_bylevel': 1,
 'colsample_bynode': 1,
 'reg_alpha': 6.345561548486771,
 'reg_lambda': 2.808394786832976,
 'scale_pos_weight': 1,
 'base_score': 0.5,
 'importance_type': 'gain',
 'num_round': 223}


In [ ]:
x_trn,x_val,y_trn,y_val = train_test_split(train,attr.a4,test_size=1000)
m = xgb.XGBRegressor().set_params(**best_hp)
m.fit(x_trn,y_trn)

In [ ]:
# ????
joblib.dump(m,f'model/tp3_{attribute}.m')

## A5

In [ ]:
attribute='a5'

In [ ]:
df = train.join(attr[attribute])

In [ ]:
best_hp={'max_depth': 4,
 'learning_rate': 0.1,
 'n_estimators': 749,
 'verbosity': 1,
 'objective': 'reg:squarederror',
 'booster': 'dart',
 'n_jobs': -1,
 'gamma': 0,
 'min_child_weight': 1,
 'max_delta_step': 0,
 'subsample': 1,
 'colsample_bytree': 1,
 'colsample_bylevel': 1,
 'colsample_bynode': 1,
 'reg_alpha': 7.819025434063891,
 'reg_lambda': 0.005996946163704,
 'scale_pos_weight': 1,
 'base_score': 0.5,
 'importance_type': 'gain',
 'num_round': 934}

In [ ]:
x_trn,x_val,y_trn,y_val = train_test_split(train,attr.a5,test_size=1000)
m = xgb.XGBRegressor().set_params(**best_hp)
m.fit(x_trn,y_trn)

In [ ]:
# ????
joblib.dump(m,f'model/tp3_{attribute}.m')

## A6 

In [ ]:
attribute='a6'

In [ ]:
df = train.join(attr[attribute])

In [ ]:
best_hp={'max_depth': 4,
 'learning_rate': 0.1,
 'n_estimators': 731,
 'verbosity': 1,
 'objective': 'reg:squarederror',
 'booster': 'dart',
 'n_jobs': -1,
 'gamma': 0,
 'min_child_weight': 1,
 'max_delta_step': 0,
 'subsample': 1,
 'colsample_bytree': 1,
 'colsample_bylevel': 1,
 'colsample_bynode': 1,
 'reg_alpha': 9.299861941127418,
 'reg_lambda': 8.762447153395918,
 'scale_pos_weight': 1,
 'base_score': 0.5,
 'importance_type': 'gain',
 'num_round': 917}

In [ ]:
x_trn,x_val,y_trn,y_val = train_test_split(train,attr.a6,test_size=1000)
m = xgb.XGBRegressor().set_params(**best_hp)
m.fit(x_trn,y_trn)

In [ ]:
# ????
joblib.dump(m,f'model/tp3_{attribute}.m')

# attribute stacking

In [ ]:
# stacking ?? attr ??
cv = KFold(6,shuffle=True,random_state=1)
attrs = [f'a{i}' for i in range(4, 7)]
trn_res = None
tst_res = None
for attribute in attrs:
    print(attribute)
    m = joblib.load(f'model/tp3_{attribute}.m')
    trn_attr = None
    tst_attr = None
    for index, (trn_idx, val_idx) in enumerate(cv.split(train, attr[attribute])):
        x_trn, x_val = train.iloc[trn_idx], train.iloc[val_idx]
        y_trn, y_val = attr[attribute].iloc[trn_idx], attr[attribute].iloc[val_idx]
        m.fit(x_trn, y_trn)
        if trn_attr is None:
            trn_attr = np.c_[val_idx, m.predict(x_val)]
            tst_attr = m.predict(test)/cv.n_splits
        else:
            trn_attr = np.r_[trn_attr, np.c_[val_idx, m.predict(x_val)]]
            tst_attr += m.predict(test)/cv.n_splits
    trn_attr = pd.DataFrame(trn_attr).set_index(0)
    trn_attr.index = trn_attr.index.astype('int')
    trn_attr.columns = [attribute]
    if trn_res is None:
        trn_res = trn_attr
        tst_res = tst_attr
    else:
        trn_res = trn_res.join(trn_attr)
        tst_res = np.c_[tst_res,tst_attr]

In [ ]:
train = train.join(trn_res)
tst_res = pd.DataFrame(tst_res)
tst_res.columns = [f'a{i}' for i in range(4,7)]
test = test.join(tst_res)

In [ ]:
# save data
save_path = 'data/tp6.h5'
train.to_hdf(save_path,'train')
target.to_hdf(save_path,'target')
test.to_hdf(save_path,'test')
attr.to_hdf(save_path,'attr')
group.to_hdf(save_path,'group')

# predict Quality_label

In [ ]:
# load data
load_path = 'data/tp6.h5'
train = pd.read_hdf(load_path,'train')
target = pd.read_hdf(load_path,'target')
test = pd.read_hdf(load_path,'test')
attr = pd.read_hdf(load_path,'attr')
group = pd.read_hdf(load_path,'group')

## FE

In [ ]:
def create_uid(df,cols,uid_name):
    df[uid_name]=df[cols[0]].astype(str)
    if len(cols)>1:
        for c in cols[1:]:
            df[uid_name]=df[uid_name]+df[c].astype(str)
    return df

def uid_aggregation(train,test,main_cols,uid,agg_method):
    for main_col in main_cols:
        new_col_name=uid+'_'+main_col+'_'+agg_method
        train[new_col_name]=train.groupby([uid])[main_col].transform(agg_method)
        test[new_col_name]=test.groupby([uid])[main_col].transform(agg_method)

def frequency_encoding(train,test,cols):
    for c in cols:
        new_col_name=c+'_freq'
        train[new_col_name] = train[c].map(train[c].value_counts())
        test[new_col_name] = test[c].map(test[c].value_counts())

def nunique_encoding(train,test,main_cols,col):
    for main_col in main_cols:
        new_col_name=main_col+'_'+col+'_nunique'
        train[new_col_name]= train[col].map(train.groupby(col)[main_col].nunique())
        test[new_col_name]= test[col].map(test.groupby(col)[main_col].nunique())

In [ ]:
for c in range(4,7):
    col='a'+str(c)
    train[col]=train[col].round(3)
    test[col]=test[col].round(3)

# creat uids
for key,value in {'uid1':['p5','p6'],'uid2':['p7','p9','p10'],'uid3':['p5','p6','p7','p8','p9'],'uid4':['p10', 'p9', 'p8', 'p7']}.items():
    train=create_uid(train,cols=value,uid_name=key)
    test=create_uid(test,cols=value,uid_name=key)
    
# apply uid_aggregation
uid_aggregation(train,test,main_cols=['a4','a5'],uid='p7',agg_method='mean')
uid_aggregation(train,test,main_cols=['a4','a5','a6'],uid='uid4',agg_method='mean')
uid_aggregation(train,test,main_cols=['a4'],uid='uid4',agg_method='std')
uid_aggregation(train,test,main_cols=['p5','p6'],uid='uid4',agg_method='sum')

# apply frequency_encoding
frequency_encoding(train,test,cols=['p7','p8','p10'])

# apply unique_encoding
nunique_encoding(train,test,main_cols=['p8','p9','p10'],col='p7')
nunique_encoding(train,test,main_cols=['p9'],col='p8')
nunique_encoding(train,test,main_cols=['uid2','uid3'],col='uid1')
nunique_encoding(train,test,main_cols=['uid3'],col='uid2')

# a4+a5+a6
train['a4_plus_a5_plus_a6']=train['a4']+train['a5']+train['a6']
test['a4_plus_a5_plus_a6']=test['a4']+test['a5']+test['a6']

# majic_fe
train['majic_fe']=train['uid4_p5_sum']*train['uid4_p6_sum']
test['majic_fe']=test['uid4_p5_sum']*test['uid4_p6_sum']

# drop_cols
drop_cols=['uid1','uid2','uid3','uid4','uid4_p5_sum','uid4_p6_sum']
train =train.drop(columns=drop_cols)
test =test.drop(columns=drop_cols)

In [ ]:
best_hp={'max_depth': 3,
 'learning_rate': 0.1,
 'n_estimators': 203,
 'verbosity': 1,
 'objective': 'multi:softmax',
 'booster': 'dart',
 'n_jobs': -1,
 'gamma': 0,
 'min_child_weight': 1,
 'max_delta_step': 0,
 'subsample': 1,
 'colsample_bytree': 1,
 'colsample_bylevel': 1,
 'colsample_bynode': 1,
 'reg_alpha': 6.736871781492226,
 'reg_lambda': 0.073512560432491,
 'scale_pos_weight': 1,
 'base_score': 0.5,
 'num_class': 4,
 'num_round': 295}

In [ ]:
from sklearn.metrics import log_loss, accuracy_score

oof = np.zeros((train.shape[0], 4))
prediction = np.zeros((test.shape[0], 4))
seeds = [123456, 87654, 676767]
num_model_seed = 3
for model_seed in range(num_model_seed):
    print("?????%d???:" % (model_seed + 1))
    oof_cat = np.zeros((train.shape[0], 4))
    prediction_cat = np.zeros((test.shape[0], 4))
    skf = StratifiedKFold(n_splits=5, random_state=seeds[model_seed], shuffle=True)
    for index, (train_index, test_index) in enumerate(skf.split(train, target)):
        print("?????%d?" % (index + 1))
        train_x, test_x, train_y, test_y = train.iloc[train_index], train.iloc[test_index], target.iloc[train_index], \
                                           target.iloc[test_index]
        gc.collect()
        m = xgb.XGBClassifier().set_params(**best_hp).set_params(n_jobs=-1,learning_rate=0.05)
        m.fit(train_x, train_y) # xgb????????
        oof_cat[test_index] += m.predict_proba(test_x)
        prediction_cat += m.predict_proba(test) / skf.n_splits
        gc.collect()
    oof += oof_cat / num_model_seed
    prediction += prediction_cat / num_model_seed
    print('logloss', log_loss(pd.get_dummies(target).values, oof_cat))
    print('ac', accuracy_score(target, np.argmax(oof_cat, axis=1)))
print("*" * 30)
print('mean_logloss', log_loss(pd.get_dummies(target).values, oof))
print('mean_ac', accuracy_score(target, np.argmax(oof, axis=1)))

group_size=50
mname='CCF_model_xgb'
res_prob = pd.DataFrame(np.c_[group,prediction])
res_prob.columns=['Group', 'Fail ratio', 'Pass ratio', 'Good ratio', 'Excellent ratio']
res_prob = res_prob[['Group', 'Excellent ratio','Good ratio', 'Pass ratio','Fail ratio']]
res_prob =(res_prob.groupby('Group').sum()/group_size).reset_index()
res_prob.to_csv(f'submission/{mname}_prob.csv',index=False,encoding='utf-8')

In [ ]:
len(train.columns)

In [ ]:
train.columns